In [1]:
from dotenv import load_dotenv
load_dotenv(".env.local")
import os
from openai import OpenAI
from pinecone import Pinecone

In [2]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [3]:
import json
data = json.load(open("injuries.json"))
data

[{'name': 'Costochondritis',
  'description': 'Inflammation of the cartilage connecting the ribs to the sternum, often triggered by heavy bench pressing, chest flyes, or high-volume pressing work. Pain is usually sharp, localized, and worsens with pressing, deep breathing, or pressing on the chest. Important: chest pain can also indicate serious cardiac issues, so this should only be considered after cardiac causes have been ruled out by a doctor.',
  'anatomy': 'Costochondral joints (where ribs meet the sternum via cartilage).',
  'treatment': 'Rest from pressing movements, anti-inflammatory measures as advised by a doctor, gradual return to pressing once pain-free. Seek immediate medical attention for chest pain accompanied by shortness of breath, radiating arm pain, or dizziness, as these can indicate a cardiac emergency.'},
 {'name': 'Rotator Cuff Tendinopathy',
  'description': 'Irritation or degeneration of the rotator cuff tendons, often caused by repetitive overhead pressing or

In [4]:
processed_data = []
client = OpenAI()

for injury in data:
    response = client.embeddings.create(
        input=f"{injury['name']}: {injury['description']}: {injury['anatomy']}: {injury['treatment']}",
        model="text-embedding-3-small",
    )
    embedding = response.data[0].embedding
    processed_data.append({
        "values": embedding,
        "id": injury["name"],
        "metadata": {
            "description": injury["description"],
            "anatomy": injury["anatomy"],
            "treatment": injury["treatment"]
        }
    })

In [5]:
processed_data[0]

{'values': [-0.016265869140625,
  -0.001682281494140625,
  0.047576904296875,
  0.02362060546875,
  0.017791748046875,
  0.047637939453125,
  -0.014312744140625,
  -0.025054931640625,
  -0.01218414306640625,
  -0.044708251953125,
  0.00469970703125,
  0.056060791015625,
  -0.0021266937255859375,
  0.0147857666015625,
  0.007495880126953125,
  0.00568389892578125,
  -0.041717529296875,
  -0.036376953125,
  -0.0369873046875,
  -0.0012178421020507812,
  0.0166015625,
  0.0282745361328125,
  0.0020084381103515625,
  -0.025543212890625,
  0.0244903564453125,
  0.01499176025390625,
  0.04937744140625,
  -0.0312042236328125,
  0.039886474609375,
  -0.005031585693359375,
  -0.005619049072265625,
  -0.0162353515625,
  -0.0004115104675292969,
  -0.0100250244140625,
  -0.00968170166015625,
  0.021820068359375,
  0.004779815673828125,
  0.0110626220703125,
  0.0030612945556640625,
  0.0029239654541015625,
  0.0206756591796875,
  0.0582275390625,
  -0.0100555419921875,
  0.0229034423828125,
  -0.03

In [6]:
index = pc.Index(os.getenv("PINECONE_INDEX"))
index.upsert(
    vectors=processed_data,
    namespace="ns1"
)

UpsertResponse(upserted_count=4)

In [7]:
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1536, total_vector_count=4, metric='cosine', namespaces=1)